# Getting Started with py-iku

This notebook introduces the fundamentals of **py-iku**, a Python library that converts
pandas/numpy data processing code into Dataiku DSS recipes, flows, and visual diagrams.

We will cover:
1. Importing the library
2. Basic rule-based conversion with `convert()`
3. Understanding the output: `DataikuFlow`, `DataikuRecipe`, `DataikuDataset`
4. Iterating over flows
5. Flow summaries with `get_summary()`
6. Visualization in ASCII and Mermaid formats
7. Serialization round-trips (`to_dict`/`from_dict`, `to_json`/`from_json`)
8. Converting from a `.py` file with `convert_file()`
9. Practical examples

---
## What py-iku Does (in 3 Cells)

py-iku takes your everyday Python data processing code and converts it into an optimized Dataiku DSS flow.
Here is the **before** (Python code) and **after** (Dataiku flow diagram) in action.

In [ ]:
# BEFORE: A typical Python data processing script
from py2dataiku import convert

python_code = """
import pandas as pd

orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')

# Clean the data
orders = orders.dropna()
orders['amount'] = orders['amount'].astype(float)

# Join orders with customer info
merged = pd.merge(orders, customers, on='customer_id')

# Summarize by region
summary = merged.groupby('region').agg({'amount': 'sum', 'order_id': 'count'})

# Get the top 10 regions
top = summary.nlargest(10, 'amount')
"""

# AFTER: One call to convert() produces an optimized Dataiku flow
demo_flow = convert(python_code, optimize=True)

print(f"Python script -> {len(demo_flow.recipes)} Dataiku recipes, {len(demo_flow.datasets)} datasets")
print(f"Recipe types: {[r.recipe_type.value for r in demo_flow.recipes]}")
print()
print(demo_flow.get_summary())

In [ ]:
# Visualize the resulting Dataiku flow as an ASCII diagram
print(demo_flow.visualize(format="ascii"))

---
## 1. Importing the Library

The main entry points are the `convert()` and `convert_file()` functions, plus the
core model classes `DataikuFlow`, `DataikuRecipe`, and `DataikuDataset`.

In [ ]:
from py2dataiku import convert, convert_file, DataikuFlow
from py2dataiku import DataikuRecipe, RecipeType
from py2dataiku import DataikuDataset, DatasetType

print("py-iku imported successfully")

---
## 2. Your First Conversion

The `convert()` function takes a Python code string and returns a `DataikuFlow`.
It uses rule-based AST pattern matching to identify pandas operations and map
them to Dataiku recipe types.

In [ ]:
# A simple pandas pipeline: read CSV and drop missing values
code = """
import pandas as pd
df = pd.read_csv('sales.csv')
df = df.dropna()
"""

flow = convert(code)
print(type(flow))  # DataikuFlow
print(flow)

The `convert()` function returns a `DataikuFlow` object, which is the main output
of py-iku. It contains datasets (data nodes) and recipes (transformation nodes)
that form a directed acyclic graph (DAG).

---
## 3. Understanding DataikuFlow

A `DataikuFlow` has two main collections:
- **datasets**: the data containers (input, intermediate, output)
- **recipes**: the transformation steps between datasets

In [ ]:
# Inspect the flow's name and timestamp
print(f"Flow name: {flow.name}")
print(f"Generated: {flow.generation_timestamp}")
print(f"Number of datasets: {len(flow.datasets)}")
print(f"Number of recipes: {len(flow.recipes)}")

In [ ]:
# List all datasets
for ds in flow.datasets:
    print(f"  {ds.name} (type={ds.dataset_type.value})")

In [ ]:
# Datasets are categorized: input, intermediate, output
print("Input datasets:", [ds.name for ds in flow.input_datasets])
print("Intermediate datasets:", [ds.name for ds in flow.intermediate_datasets])
print("Output datasets:", [ds.name for ds in flow.output_datasets])

---
## 4. Understanding DataikuRecipe

Recipes are the transformation nodes in the flow. Each recipe has:
- A `name` and a `recipe_type` (from the `RecipeType` enum)
- `inputs` and `outputs` (lists of dataset names)
- Type-specific attributes (e.g., `steps` for PREPARE recipes)

In [ ]:
# List all recipes in the flow
for recipe in flow.recipes:
    print(f"Recipe: {recipe.name}")
    print(f"  Type: {recipe.recipe_type.value}")
    print(f"  Inputs: {recipe.inputs}")
    print(f"  Outputs: {recipe.outputs}")
    print()

---
## 5. Understanding DataikuDataset

Each dataset has:
- A `name`
- A `dataset_type` (`INPUT`, `INTERMEDIATE`, or `OUTPUT`)
- A `connection_type` (default: `FILESYSTEM`)
- Optional `schema`, `source_variable`, and `source_line`

In [ ]:
# Inspect the first dataset in detail
if flow.datasets:
    ds = flow.datasets[0]
    print(f"Name: {ds.name}")
    print(f"Dataset type: {ds.dataset_type}")
    print(f"Connection type: {ds.connection_type}")
    print(f"Is input: {ds.is_input}")
    print(f"Is output: {ds.is_output}")

In [ ]:
# Look up a dataset by name
found = flow.get_dataset(flow.datasets[0].name)
print(f"Found dataset: {found}")

---
## 6. Iterating Over a Flow

`DataikuFlow` supports Python's iteration protocol:
- `len(flow)` returns the number of recipes
- `for recipe in flow` iterates over recipes

In [ ]:
# len() returns the recipe count
print(f"Number of recipes: {len(flow)}")

In [ ]:
# Iterate over recipes using a for loop
for i, recipe in enumerate(flow):
    print(f"Recipe {i}: {recipe.name} ({recipe.recipe_type.value})")

---
## 7. Flow Summary

The `get_summary()` method provides a text overview of the entire flow,
including dataset counts, recipe counts by type, and optimization notes.

In [ ]:
print(flow.get_summary())

---
## 8. Visualization

py-iku can visualize flows in multiple formats. The `flow.visualize()` method
accepts a `format` parameter. Here we demonstrate **ASCII**, **SVG**, and **Mermaid**.

### 8.1 ASCII Visualization

Great for quick terminal-friendly output.

In [ ]:
ascii_output = flow.visualize(format="ascii")
print(ascii_output)

### 8.2 SVG Visualization (Jupyter Inline)

When a `DataikuFlow` is the last expression in a Jupyter cell, it renders inline
via `_repr_svg_()`. You can also use `flow.visualize(format="svg")` explicitly.

In [ ]:
# SVG renders inline in Jupyter via _repr_svg_()
# Simply evaluate the flow as the last expression in a cell:
flow

### 8.3 Mermaid Visualization

Mermaid diagrams can be rendered in GitHub, Notion, and many Markdown viewers.

In [ ]:
mermaid_output = flow.visualize(format="mermaid")
print(mermaid_output)

### 8.3 PlantUML Visualization

PlantUML diagrams for technical documentation and UML tools.

In [ ]:
plantuml_output = flow.visualize(format='plantuml')
print(plantuml_output)

---
## 9. Serialization Round-Trips

Flows can be serialized to dictionaries, JSON, or YAML, and reconstructed
back. This is useful for saving, sharing, and versioning flow definitions.

### 9.1 to_dict / from_dict

In [ ]:
# Serialize to a dictionary
flow_dict = flow.to_dict()
print(f"Dict keys: {list(flow_dict.keys())}")
print(f"Flow name from dict: {flow_dict['flow_name']}")
print(f"Total recipes: {flow_dict['total_recipes']}")
print(f"Total datasets: {flow_dict['total_datasets']}")

In [ ]:
# Reconstruct from dictionary
flow_restored = DataikuFlow.from_dict(flow_dict)
print(f"Restored flow: {flow_restored}")
print(f"Recipes match: {len(flow_restored) == len(flow)}")

### 9.2 to_json / from_json

In [ ]:
# Serialize to JSON string
json_str = flow.to_json(indent=2)
print(json_str[:300])  # Print the first 300 chars
print("...")

In [ ]:
# Reconstruct from JSON string
flow_from_json = DataikuFlow.from_json(json_str)
print(f"From JSON: {flow_from_json}")
print(f"Names match: {flow_from_json.name == flow.name}")

---
## 10. Converting from a File

The `convert_file()` function reads a `.py` file and converts it.
Let's write a temporary Python file and convert it.

In [ ]:
import tempfile
import os

# Write a temporary Python file
pipeline_code = """
import pandas as pd

df = pd.read_csv('customers.csv')
df = df.dropna()
df = df.sort_values('signup_date')
df.to_csv('customers_clean.csv', index=False)
"""

tmp_file = os.path.join(tempfile.gettempdir(), "my_pipeline.py")
with open(tmp_file, "w") as f:
    f.write(pipeline_code)

print(f"Written to: {tmp_file}")

In [ ]:
# Convert from file
file_flow = convert_file(tmp_file)

print(f"Source file: {file_flow.source_file}")
print()
print(file_flow.get_summary())

In [ ]:
# Clean up the temporary file
os.remove(tmp_file)

---
## 11. Practical Examples

Below are several simple pandas patterns and their Dataiku flow equivalents.

### Example 1: Read CSV

`pd.read_csv()` creates an input dataset in the flow.

In [ ]:
flow_read = convert("""
import pandas as pd
df = pd.read_csv('orders.csv')
""")

print(f"Datasets: {[ds.name for ds in flow_read.datasets]}")
print(f"Recipes: {len(flow_read)}")
print(flow_read.visualize(format="ascii"))

### Example 2: Drop Missing Values (dropna)

`df.dropna()` maps to a PREPARE recipe with a REMOVE_ROWS_ON_EMPTY processor.

In [ ]:
flow_dropna = convert("""
import pandas as pd
df = pd.read_csv('data.csv')
df = df.dropna()
""")

for recipe in flow_dropna:
    print(f"Recipe: {recipe.name} (type={recipe.recipe_type.value})")
    if recipe.steps:
        for step in recipe.steps:
            print(f"  Step: {step.processor_type.value}")

print()
print(flow_dropna.visualize(format="ascii"))

### Example 3: Sort Values

`df.sort_values()` maps to a SORT recipe.

In [ ]:
flow_sort = convert("""
import pandas as pd
df = pd.read_csv('products.csv')
df = df.sort_values('price', ascending=False)
""")

for recipe in flow_sort:
    print(f"Recipe: {recipe.name} (type={recipe.recipe_type.value})")

print()
print(flow_sort.visualize(format="ascii"))

### Example 4: Group By with Aggregation

`df.groupby().agg()` maps to a GROUPING recipe.

In [ ]:
flow_group = convert("""
import pandas as pd
df = pd.read_csv('sales.csv')
result = df.groupby('region').agg({'revenue': 'sum'})
""")

for recipe in flow_group:
    print(f"Recipe: {recipe.name} (type={recipe.recipe_type.value})")
    if recipe.recipe_type == RecipeType.GROUPING:
        print(f"  Group keys: {recipe.group_keys}")
        for agg in recipe.aggregations:
            print(f"  Aggregation: {agg.function}({agg.column})")

print()
print(flow_group.visualize(format="ascii"))

### Example 5: Drop Duplicates

`df.drop_duplicates()` maps to a PREPARE recipe with deduplication processing.

In [ ]:
flow_distinct = convert("""
import pandas as pd
df = pd.read_csv('emails.csv')
df = df.drop_duplicates()
""")

for recipe in flow_distinct:
    print(f"Recipe: {recipe.name} (type={recipe.recipe_type.value})")

print()
print(flow_distinct.visualize(format="ascii"))

### Example 6: Rename Columns

`df.rename(columns=...)` maps to a PREPARE recipe with a COLUMN_RENAMER processor.

In [ ]:
flow_rename = convert("""
import pandas as pd
df = pd.read_csv('users.csv')
df = df.rename(columns={'old_name': 'new_name'})
""")

for recipe in flow_rename:
    print(f"Recipe: {recipe.name} (type={recipe.recipe_type.value})")
    if recipe.steps:
        for step in recipe.steps:
            print(f"  Step: {step.processor_type.value} - params: {step.params}")

print()
print(flow_rename.visualize(format="ascii"))

---
## 12. Combining Multiple Operations

Real pipelines chain multiple operations. py-iku handles multi-step
code and produces a flow with multiple recipes.

In [ ]:
multi_step_code = """
import pandas as pd

# Load data
df = pd.read_csv('transactions.csv')

# Clean: remove missing values
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates()

# Sort by date
df = df.sort_values('transaction_date')

# Aggregate by category
summary = df.groupby('category').agg({'amount': 'sum'})
"""

flow_multi = convert(multi_step_code)
print(flow_multi.get_summary())

In [ ]:
# Visualize the multi-step flow in ASCII
print(flow_multi.visualize(format="ascii"))

In [ ]:
# Visualize the multi-step flow in Mermaid
print(flow_multi.visualize(format="mermaid"))

---
## 13. Inspecting Recipe Details

You can look up recipes by name or by type, and inspect their
internal configuration.

In [ ]:
# Look up a recipe by name
if flow_multi.recipes:
    first_recipe = flow_multi.recipes[0]
    found_recipe = flow_multi.get_recipe(first_recipe.name)
    print(f"Found recipe by name: {found_recipe}")

In [ ]:
# Find recipes by type
grouping_recipes = flow_multi.get_recipes_by_type(RecipeType.GROUPING)
print(f"Grouping recipes: {len(grouping_recipes)}")
for r in grouping_recipes:
    print(f"  {r.name}: keys={r.group_keys}")

---
## 14. Converting Recipes to Dictionaries

Individual recipes can also be serialized for inspection or export.

In [ ]:
import json

# Serialize a single recipe to a dictionary
if flow_multi.recipes:
    recipe_dict = flow_multi.recipes[0].to_dict()
    print(json.dumps(recipe_dict, indent=2))

---
## Summary

In this notebook you learned:

- **`convert(code)`** analyzes a Python code string using rule-based AST matching and returns a `DataikuFlow`.
- **`DataikuFlow`** holds `datasets` and `recipes`, supports `len()`, iteration, and `get_summary()`.
- **`DataikuRecipe`** represents a transformation node with a `recipe_type`, `inputs`, `outputs`, and type-specific fields.
- **`DataikuDataset`** represents a data node with a `dataset_type` (INPUT, INTERMEDIATE, OUTPUT).
- **Visualization**: `flow.visualize(format="ascii")` for terminal output, `flow.visualize(format="mermaid")` for Markdown-compatible diagrams.
- **Serialization**: `to_dict()`/`from_dict()` and `to_json()`/`from_json()` enable round-trip persistence.
- **`convert_file(path)`** reads a `.py` file and converts it, setting `flow.source_file`.

**Common pandas-to-Dataiku mappings covered:**

| pandas operation | Dataiku recipe type |
|---|---|
| `pd.read_csv()` | Input dataset |
| `df.dropna()` | PREPARE (REMOVE_ROWS_ON_EMPTY) |
| `df.sort_values()` | SORT |
| `df.groupby().agg()` | GROUPING |
| `df.drop_duplicates()` | PREPARE (deduplication) |
| `df.rename(columns=...)` | PREPARE (COLUMN_RENAMER) |

Next notebook: **02_intermediate.ipynb** covers more recipe types, join operations, and advanced visualization formats.

---
## 15. Deploying to Dataiku DSS

py-iku can deploy flows directly to a Dataiku DSS instance using `DSSFlowDeployer`.
In **dry-run mode**, the deployer validates the flow and shows what *would* be created
without making any API calls -- no DSS connection or `dataikuapi` package required.

In [ ]:
from py2dataiku.integrations import DSSFlowDeployer

# Create a deployer in dry-run mode (no DSS connection required)
deployer = DSSFlowDeployer(
    host="https://dss.example.com",
    api_key="dummy-api-key",
    project_key="BEGINNER_DEMO",
    dry_run=True,
)

# Deploy the demo_flow we created at the top of this notebook
result = deployer.deploy(demo_flow)
print(result)
print(f"\nSuccess: {result.success}")
print(f"Datasets that would be created: {result.datasets_created}")
print(f"Recipes that would be created: {result.recipes_created}")

In [ ]:
# Inspect the deployment result in detail
print("Deployment Result Details:")
print(f"  Dry run: {result.dry_run}")
print(f"  Warnings: {result.warnings}")
print(f"  Errors: {result.errors}")
print()

# The result can also be serialized to a dictionary
result_dict = result.to_dict()
print("Result as dict:")
for key, value in result_dict.items():
    print(f"  {key}: {value}")

**What is dry-run mode?**

When `dry_run=True`, the deployer:
- Validates the flow structure (checks for cycles, missing connections)
- Determines the topological order for creating datasets and recipes
- Returns a `DeploymentResult` listing what *would* be created
- Does **not** make any network calls or require `dataikuapi` to be installed

This is useful for previewing a deployment before connecting to a live DSS instance.
To deploy for real, set `dry_run=False` and provide a valid DSS `host` and `api_key`.